# Tutorial 3: The $\rho$-Scaled Optimizer

This notebook demonstrates `RhoScaledAdam`, an optimizer that
automatically scales the learning rate by the convergence radius
$\rho = \pi / \Delta$.

When predictions sharpen ($\Delta$ grows), $\rho$ shrinks, and the
optimizer reduces the effective LR to stay within the safe region.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from ghosts.control import RhoScaledAdam, computeRho
from ghosts.radii import compute_rho_out

## 1. Basic usage

Pass logits to `opt.step(logits=...)` so the optimizer can compute $\rho$.

In [ ]:
model = nn.Linear(8, 4)
opt = RhoScaledAdam(model.parameters(), lr=0.01, rhoCap=1.0)

x = torch.randn(4, 8)
targets = torch.tensor([0, 1, 2, 3])

logits = model(x)
loss = nn.functional.cross_entropy(logits, targets)
opt.zero_grad()
loss.backward()
opt.step(logits=logits)

stats = opt.getStats()
print(f"rho = {stats['rho']:.4f}")
print(f"base LR = {stats['baseLR']}")
print(f"effective LR = {stats['effectiveLR']:.6f}")
print(f"scale = {stats['scale']:.4f}")

## 2. Comparing Adam vs RhoScaledAdam

We train a small model on random data and track loss + $\rho$ over time.

In [ ]:
torch.manual_seed(42)
nSamples, nFeatures, nClasses = 64, 16, 5
X = torch.randn(nSamples, nFeatures)
Y = torch.randint(0, nClasses, (nSamples,))

def trainLoop(optClass, lr, steps=200, **kwargs):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(nFeatures, 32), nn.ReLU(), nn.Linear(32, nClasses))
    if optClass == RhoScaledAdam:
        opt = optClass(net.parameters(), lr=lr, **kwargs)
    else:
        opt = optClass(net.parameters(), lr=lr)
    losses, rhos, effLRs = [], [], []
    for _ in range(steps):
        logits = net(X)
        loss = nn.functional.cross_entropy(logits, Y)
        opt.zero_grad()
        loss.backward()
        if optClass == RhoScaledAdam:
            opt.step(logits=logits)
            s = opt.getStats()
            rhos.append(s['rho'])
            effLRs.append(s['effectiveLR'])
        else:
            opt.step()
            rho = compute_rho_out(logits.detach(), gap='maxmin', reduce='mean')
            rhos.append(rho)
            effLRs.append(lr)
        losses.append(loss.item())
    return losses, rhos, effLRs

In [ ]:
lr = 0.01
lAdam, rAdam, eAdam = trainLoop(torch.optim.Adam, lr)
lRho, rRho, eRho = trainLoop(RhoScaledAdam, lr, rhoCap=1.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(lAdam, label='Adam')
axes[0].plot(lRho, label=r'$\rho$-Adam')
axes[0].set_ylabel('Loss')
axes[0].set_xlabel('Step')
axes[0].legend()
axes[0].set_title('Training loss')

axes[1].plot(rAdam, label='Adam')
axes[1].plot(rRho, label=r'$\rho$-Adam')
axes[1].set_ylabel(r'$\rho$')
axes[1].set_xlabel('Step')
axes[1].legend()
axes[1].set_title('Convergence radius')

axes[2].plot(eAdam, label='Adam')
axes[2].plot(eRho, label=r'$\rho$-Adam')
axes[2].set_ylabel('Effective LR')
axes[2].set_xlabel('Step')
axes[2].legend()
axes[2].set_title('Effective learning rate')

plt.tight_layout()
plt.show()

## 3. Effect of `rhoCap` and `rhoFloor`

`rhoCap` limits the maximum scale (prevents too-large steps early in training).
`rhoFloor` prevents the scale from going to zero (avoids stalling).

In [ ]:
configs = [
    {"rhoCap": 1.0, "rhoFloor": 0.01, "label": "cap=1.0, floor=0.01"},
    {"rhoCap": 0.5, "rhoFloor": 0.01, "label": "cap=0.5, floor=0.01"},
    {"rhoCap": 1.0, "rhoFloor": 0.1, "label": "cap=1.0, floor=0.1"},
]

fig, ax = plt.subplots(figsize=(6, 4))
for cfg in configs:
    losses, _, _ = trainLoop(
        RhoScaledAdam, 0.01,
        rhoCap=cfg["rhoCap"], rhoFloor=cfg["rhoFloor"],
    )
    ax.plot(losses, label=cfg["label"])

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title(r'Effect of $\rho$ cap and floor')
ax.legend()
plt.tight_layout()
plt.show()